# Align Miniscope To ezTrack Batch

This notebook batch-aligns miniscope timestamps to ezTrack behavior-camera tracking outputs for all discovered sessions from a given mouse.

## 1. Imports

In [1]:
from pathlib import Path
import glob
import re

import numpy as np
import pandas as pd
from tqdm import tqdm


## 2. User Inputs

In [7]:
# behavior analysis info
behavTimeStampsBaseDir = "/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackEzTrack"
mouse = 752

# optional export settings
save_outputs = True
output_dir = None  # e.g. Path(behavTimeStampsBaseDir) / f"aligned_miniscope_to_eztrack_{mouse}"


## 3. Helper Functions

In [8]:
def discover_mouse_sessions(base_dir, mouse):
    mouse = str(mouse)
    base_dir = Path(base_dir)
    session_dir = base_dir / f"BehavCamConcactenated_{mouse}"
    pattern = re.compile(rf"^m{mouse}_(\d{{8}}_\d{{2}}_\d{{2}}_\d{{2}})_concactenatedbehavCam.*\.mp4$")

    rows = []
    for video_path in sorted(session_dir.glob("m*.mp4")):
        match = pattern.match(video_path.name)
        if match is None:
            continue
        session = match.group(1)
        rows.append({
            "mouse": int(mouse),
            "session": session,
            "source_video": video_path,
        })

    return pd.DataFrame(rows)


def resolve_behavior_files(base_dir, mouse, session):
    mouse = str(mouse)
    base_dir = Path(base_dir)
    date_part, time_part = session.split("_", 1)

    candidate_dates = [
        ("MMDDYYYY", date_part[:2], date_part[2:4], date_part[4:8]),
        ("DDMMYYYY", date_part[2:4], date_part[:2], date_part[4:8]),
    ]

    tried = []
    for fmt_name, mm, dd, yyyy in candidate_dates:
        ymd_session = f"{yyyy}_{mm}_{dd}_{mouse}_{time_part}"
        source_video_pattern = str(
            base_dir / f"BehavCamConcactenated_{mouse}" /
            f"m{mouse}_{session}_concactenatedbehavCam*.mp4"
        )
        eztrack_pattern = str(
            base_dir / f"BehavCamConcactenated_{mouse}" / "rotated_and_cropped_avi" /
            f"m{mouse}_{session}_*_LocationOutput.csv"
        )
        behav_ts_pattern = str(
            base_dir / f"timeStampsBehavCam_copied_{mouse}" /
            f"{ymd_session}_timeStampsBehavCam.csv"
        )
        miniscope_ts_pattern = str(
            base_dir / f"timeStampsMiniscope_copied_{mouse}" /
            f"{ymd_session}_timeStampsMiniscope.csv"
        )

        source_video_matches = sorted(glob.glob(source_video_pattern))
        eztrack_matches = sorted(glob.glob(eztrack_pattern))
        behav_matches = sorted(glob.glob(behav_ts_pattern))
        miniscope_matches = sorted(glob.glob(miniscope_ts_pattern))

        tried.append({
            "format": fmt_name,
            "ymd_session": ymd_session,
            "source_video_matches": source_video_matches,
            "eztrack_matches": eztrack_matches,
            "behav_matches": behav_matches,
            "miniscope_matches": miniscope_matches,
        })

        if len(source_video_matches) == 1 and len(eztrack_matches) == 1 and len(behav_matches) == 1 and len(miniscope_matches) == 1:
            return {
                "date_format_used": fmt_name,
                "source_video": Path(source_video_matches[0]),
                "ezTrackOutput": Path(eztrack_matches[0]),
                "timestampfile": Path(behav_matches[0]),
                "miniscope_timestampfile": Path(miniscope_matches[0]),
            }

    debug_lines = []
    for attempt in tried:
        debug_lines.append(
            f"{attempt['format']} -> {attempt['ymd_session']}\n"
            f"  source video: {len(attempt['source_video_matches'])} match(es)\n"
            f"  ezTrack: {len(attempt['eztrack_matches'])} match(es)\n"
            f"  behav: {len(attempt['behav_matches'])} match(es)\n"
            f"  miniscope: {len(attempt['miniscope_matches'])} match(es)"
        )

    raise FileNotFoundError(
        "Could not uniquely resolve files for session.\n\n" + "\n\n".join(debug_lines)
    )


def rounded_fps_from_timestamps(timestamp_df):
    return round((1000 / timestamp_df["Time Stamp (ms)"].diff().dropna().mean()) / 5) * 5


def align_single_session(base_dir, mouse, session):
    files = resolve_behavior_files(base_dir, mouse, session)
    eztrack_df = pd.read_csv(files["ezTrackOutput"]).reset_index(drop=True)
    behav_df = pd.read_csv(files["timestampfile"]).reset_index(drop=True)
    miniscope_df = pd.read_csv(files["miniscope_timestampfile"]).reset_index(drop=True)

    behav_fps = rounded_fps_from_timestamps(behav_df)
    miniscope_fps = rounded_fps_from_timestamps(miniscope_df)

    behav_lookup = behav_df[["Time Stamp (ms)"]].copy()
    behav_lookup["closestBehavCamFrameIdx"] = np.arange(len(behav_lookup))
    behav_lookup = behav_lookup.rename(columns={"Time Stamp (ms)": "sys_clock_BehavCamFrame"})

    miniscope_aligned = miniscope_df.copy()
    miniscope_aligned["_mini_row_order"] = np.arange(len(miniscope_aligned))

    miniscope_aligned = pd.merge_asof(
        miniscope_aligned.sort_values("Time Stamp (ms)"),
        behav_lookup.sort_values("sys_clock_BehavCamFrame"),
        left_on="Time Stamp (ms)",
        right_on="sys_clock_BehavCamFrame",
        direction="nearest",
    )

    miniscope_aligned["closestBehavCamFrameIdx"] = miniscope_aligned["closestBehavCamFrameIdx"].astype(int)
    miniscope_aligned["abs_timestamp_diff_ms"] = (
        miniscope_aligned["Time Stamp (ms)"] - miniscope_aligned["sys_clock_BehavCamFrame"]
    ).abs()

    tracking_lookup = eztrack_df.reset_index().rename(columns={
        "index": "closestBehavCamFrameIdx",
        "X": "X_coor",
        "Y": "Y_coor",
    })
    tracking_cols = [col for col in ["closestBehavCamFrameIdx", "X_coor", "Y_coor", "Distance_px"] if col in tracking_lookup.columns]
    miniscope_aligned = miniscope_aligned.merge(tracking_lookup[tracking_cols], on="closestBehavCamFrameIdx", how="left")

    miniscope_aligned["mouse"] = int(mouse)
    miniscope_aligned["session"] = session
    miniscope_aligned["date_format_used"] = files["date_format_used"]
    miniscope_aligned["behav_fps"] = behav_fps
    miniscope_aligned["miniscope_fps"] = miniscope_fps
    miniscope_aligned["ezTrackOutput"] = str(files["ezTrackOutput"])
    miniscope_aligned["timestampfile"] = str(files["timestampfile"])
    miniscope_aligned["miniscope_timestampfile"] = str(files["miniscope_timestampfile"])

    miniscope_aligned = miniscope_aligned.sort_values("_mini_row_order").drop(columns="_mini_row_order").reset_index(drop=True)

    meta = {
        "mouse": int(mouse),
        "session": session,
        "date_format_used": files["date_format_used"],
        "behav_fps": behav_fps,
        "miniscope_fps": miniscope_fps,
        "n_behav_frames": len(behav_df),
        "n_miniscope_frames": len(miniscope_df),
        "n_eztrack_rows": len(eztrack_df),
        "ezTrackOutput": str(files["ezTrackOutput"]),
        "timestampfile": str(files["timestampfile"]),
        "miniscope_timestampfile": str(files["miniscope_timestampfile"]),
    }
    return miniscope_aligned, meta


def batch_align_mouse_sessions(base_dir, mouse, save_outputs=True, output_dir=None):
    base_dir = Path(base_dir)
    sessions_df = discover_mouse_sessions(base_dir, mouse)
    aligned_by_session = {}
    summary_rows = []
    combined_parts = []

    if output_dir is None:
        output_dir = base_dir / f"aligned_miniscope_to_eztrack_{mouse}"
    else:
        output_dir = Path(output_dir)

    if save_outputs:
        output_dir.mkdir(parents=True, exist_ok=True)

    for row in tqdm(sessions_df.itertuples(index=False), total=len(sessions_df), desc=f"Mouse {mouse} sessions"):
        try:
            aligned_df, meta = align_single_session(base_dir, mouse, row.session)
            aligned_by_session[row.session] = aligned_df
            combined_parts.append(aligned_df)

            output_path = None
            if save_outputs:
                output_path = output_dir / f"{row.session}_miniscopeAlignedToEzTrack.csv"
                aligned_df.to_csv(output_path, index=False)

            summary_rows.append({
                **meta,
                "status": "ok",
                "output_path": str(output_path) if output_path is not None else None,
            })
        except Exception as exc:
            summary_rows.append({
                "mouse": int(mouse),
                "session": row.session,
                "status": "error",
                "error": str(exc),
                "output_path": None,
            })

    summary_df = pd.DataFrame(summary_rows)
    combined_df = pd.concat(combined_parts, ignore_index=True, sort=False) if combined_parts else pd.DataFrame()

    if save_outputs:
        summary_df.to_csv(output_dir / f"mouse_{mouse}_alignmentSummary.csv", index=False)
        if not combined_df.empty:
            combined_df.to_csv(output_dir / f"mouse_{mouse}_allSessions_miniscopeAlignedToEzTrack.csv", index=False)

    return sessions_df, summary_df, combined_df, aligned_by_session


## 4. Discover Sessions For This Mouse

In [9]:
sessions_df = discover_mouse_sessions(behavTimeStampsBaseDir, mouse)
sessions_df


,mouse,session,source_video
0,752,01012025_18_25_40,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...
1,752,01102024_16_18_25,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...
2,752,02012025_19_56_52,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...
3,752,02102024_14_10_13,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...
4,752,03102024_13_18_31,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...
5,752,04012025_18_46_25,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...
6,752,04102024_15_56_54,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...
7,752,05012025_16_56_29,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...
8,752,06012025_20_35_19,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...
9,752,06012025_20_48_29,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...


## 5. Batch Align All Sessions

In [ ]:
sessions_df, summary_df, combined_aligned_df, aligned_by_session = batch_align_mouse_sessions(
    behavTimeStampsBaseDir,
    mouse,
    save_outputs=save_outputs,
    output_dir=output_dir,
)
summary_df

Mouse 752 sessions: 100%|███████████████████████| 16/16 [02:52<00:00, 10.76s/it]


In [6]:
summary_df['output_path'].values[0]

'/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackEzTrack/aligned_miniscope_to_eztrack_328/04062025_17_43_00_miniscopeAlignedToEzTrack.csv'

## 6. Inspect Combined Output

In [ ]:
combined_aligned_df.head()


## 7. Inspect One Session

In [ ]:
session_to_view = sessions_df.iloc[0]["session"] if not sessions_df.empty else None
aligned_by_session.get(session_to_view, pd.DataFrame()).head()
